# Music Generation with an RNN (LSTM) in PyTorch

A fully-working recurrent neural network (RNN) that learns patterns in Irish folk songs written in [ABC notation](https://en.wikipedia.org/wiki/ABC_notation), and generates new songs character-by-character, built with PyTorch by subclassing `nn.Module`.

This is an adapted version of **Lab 1, Part 2** from MIT's [Introduction to Deep Learning (6.S191)](http://introtodeeplearning.com) course. The original lab is a guided exercise with `#TODO` sections for students to fill in; this version has all TODOs completed and has been trimmed down to focus purely on the working pipeline:

- Removed Colab/copyright banners and most explanatory/demo cells so only the core data -> model -> train -> generate pipeline remains
- [Comet ML](https://www.comet.com/) integration for experiment tracking (add your own API key to use it)


**Pipeline:** load ABC-notation songs -> vectorize characters -> train an `Embedding -> LSTM -> Linear` model to predict the next character -> sample new text from the trained model -> convert generated ABC text back into audio.

Original course materials: [introtodeeplearning.com](http://introtodeeplearning.com) · [GitHub](https://github.com/MITDeepLearning/introtodeeplearning)

In [ ]:
# Core PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim

# MIT Introduction to Deep Learning helper package: provides the ABC-notation
# song dataset, audio synthesis utilities, and a live loss plotter used below
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# Other packages used for training, timing, and turning generated ABC text into audio
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write

# System tools required to synthesize audio (.wav files) from ABC notation text
!apt-get install abcmidi timidity > /dev/null 2>&1

## Comet ML (experiment tracking)

[Comet](https://www.comet.com/) logs your training metrics and generated audio to a dashboard. Paste your API key below to enable it (get it from your [Comet account settings](https://www.comet.com/api/my/settings)).

In [ ]:
# Install and import Comet ML
!pip install comet_ml --quiet
import comet_ml

# TODO: paste your Comet API key here
COMET_API_KEY = ""
assert COMET_API_KEY != "", "Please insert your Comet API Key"

In [ ]:
# Use the GPU if one is available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Load the dataset

In [ ]:
# Download the dataset: a collection of thousands of Irish folk songs in ABC notation
songs = mdl.lab1.load_training_data()

# Print one song to inspect the raw ABC notation format
example_song = songs[0]
print("Example song:\n", example_song)

# Convert the example song to an audio waveform and listen to it
mdl.lab1.play_song(example_song)

## 2. Prepare the text data for training

In [ ]:
# Join every song into one long string, separated by blank lines
songs_joined = "\n\n".join(songs)

# Find every unique character used across all songs -- this is our vocabulary
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

# Create a mapping from character -> integer index, and the reverse mapping back to characters
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

In [ ]:
def vectorize_string(string):
    # Convert every character in the string to its integer index using char2idx
    return np.array([char2idx[c] for c in string])

vectorized_songs = vectorize_string(songs_joined)

# Sanity check: should be a NumPy array with one entry per character
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"
print(repr(songs_joined[:10]), "---- characters mapped to int ---->", vectorized_songs[:10])

In [ ]:
def get_batch(vectorized_songs, seq_length, batch_size):
    # The length of the vectorized songs string
    n = vectorized_songs.shape[0] - 1
    # Randomly choose the starting indices for the examples in the training batch
    idx = np.random.choice(n - seq_length, batch_size)

    # Build input sequences: `seq_length` characters starting at each chosen index
    input_batch = [vectorized_songs[i : i + seq_length] for i in idx]

    # Build target sequences: the same characters shifted one position to the
    # right -- i.e. the "next character" the model should learn to predict
    output_batch = [vectorized_songs[i + 1 : i + seq_length + 1] for i in idx]

    # Convert the input and output batches to tensors
    x_batch = torch.tensor(input_batch, dtype=torch.long)
    y_batch = torch.tensor(output_batch, dtype=torch.long)

    return x_batch, y_batch

# Sanity check: make sure the batch function returns the expected shapes
test_args = (vectorized_songs, 10, 2)
x_batch, y_batch = get_batch(*test_args)
assert x_batch.shape == (2, 10), "x_batch shape is incorrect"
assert y_batch.shape == (2, 10), "y_batch shape is incorrect"
print("Batch function works correctly!")

## 3. Define the RNN (LSTM) model

The model is defined by subclassing `nn.Module` and is composed of three layers:

- `nn.Embedding` -- maps each character index to a trainable dense vector
- `nn.LSTM` -- processes the sequence of embeddings, maintaining a hidden state that carries information about previously seen characters
- `nn.Linear` -- maps each LSTM output to a score (logit) for every character in the vocabulary

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size

        # Layer 1: Embedding layer -- maps each character index to a dense
        # vector of size `embedding_dim` that the model learns during training
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Layer 2: LSTM layer with `hidden_size` units. batch_first=True means
        # tensors are shaped (batch_size, sequence_length, features)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)

        # Layer 3: Linear (fully-connected) layer that turns each LSTM output
        # into a score (logit) for every character in the vocabulary
        self.fc = nn.Linear(hidden_size, vocab_size)

    def init_hidden(self, batch_size, device):
        # Initialize the LSTM's hidden state and cell state with zeros
        return (torch.zeros(1, batch_size, self.hidden_size).to(device),
                torch.zeros(1, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):
        x = self.embedding(x)

        if state is None:
            state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)

        out = self.fc(out)
        return out if not return_state else (out, state)


# Instantiate the model with some default hyperparameters and move it to the device
vocab_size = len(vocab)
embedding_dim = 256
hidden_size = 1024

model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)
print(model)

# Sanity check: run one batch through the model and confirm the output shape
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
x, y = x.to(device), y.to(device)
pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

## 4. Define the loss function

In [10]:
# CrossEntropyLoss combines log-softmax + negative log-likelihood, and expects
# raw logits (not probabilities) as input
cross_entropy = nn.CrossEntropyLoss()

def compute_loss(labels, logits):
    """
    Inputs:
      labels: (batch_size, sequence_length)
      logits: (batch_size, sequence_length, vocab_size)

    Output:
      loss: scalar cross entropy loss over the batch and sequence length
    """
    # Flatten the labels so their shape is (batch_size * sequence_length,)
    batched_labels = labels.view(-1)

    # Flatten the logits to match: (batch_size * sequence_length, vocab_size)
    batched_logits = logits.view(-1, logits.size(-1))

    # Compute the cross-entropy loss between the flattened predictions and labels
    loss = cross_entropy(batched_logits, batched_labels)
    return loss

## 5. Train the model

In [11]:
vocab_size = len(vocab)

# Model & training hyperparameters -- feel free to experiment with these
params = dict(
    num_training_iterations=3000,  # increase this to train longer
    batch_size=8,                  # try values between 1 and 64
    seq_length=100,                # try values between 50 and 500
    learning_rate=5e-3,            # try values between 1e-5 and 1e-1
    embedding_dim=256,
    hidden_size=1024,              # try values between 1 and 2048
)

# Where model checkpoints will be saved during training
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")
os.makedirs(checkpoint_dir, exist_ok=True)

In [12]:
def create_experiment():
    # End any previous experiment before starting a new one
    if 'experiment' in globals() and experiment is not None:
        experiment.end()

    # Start a new Comet experiment for this training run
    experiment = comet_ml.Experiment(
        api_key=COMET_API_KEY,
        project_name="music-generation-pytorch")

    # Log our hyperparameters to Comet so this run is easy to compare against others
    for param, value in params.items():
        experiment.log_parameter(param, value)
    experiment.flush()

    return experiment

In [ ]:
# Instantiate a fresh model using the hyperparameters defined above
model = LSTMModel(vocab_size, params["embedding_dim"], params["hidden_size"]).to(device)

# Adam is a good default optimizer choice for training RNNs
optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])

def train_step(x, y):
    model.train()          # set the model to training mode
    optimizer.zero_grad()  # clear gradients from the previous step

    # Forward pass: feed the input batch into the model to get predictions
    y_hat = model(x)

    # Compute the loss between the predictions and the true next characters
    loss = compute_loss(y, y_hat)

    # Backward pass: compute gradients, then update the model's parameters
    loss.backward()
    optimizer.step()

    return loss

##################
# Begin training #
##################
history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
experiment = create_experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear()  # clear any leftover progress bars

for iter in tqdm(range(params["num_training_iterations"])):
    # Grab a random batch of input/target sequences from the training data
    x_batch, y_batch = get_batch(vectorized_songs, params["seq_length"], params["batch_size"])
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    # Take one training step and get the resulting loss
    loss = train_step(x_batch, y_batch)

    # Log the loss to Comet and to our local history for the live plot
    experiment.log_metric("loss", loss.item(), step=iter)
    history.append(loss.item())
    plotter.plot(history)

    # Periodically save a checkpoint of the model's weights
    if iter % 100 == 0:
        torch.save(model.state_dict(), checkpoint_prefix)

# Save the final trained model
torch.save(model.state_dict(), checkpoint_prefix)
experiment.flush()

## 6. Generate new music

In [ ]:
def generate_text(model, start_string, generation_length=1000):
    # Convert the start string to its numeric (vectorized) representation
    input_idx = [char2idx[s] for s in start_string]
    input_idx = torch.tensor([input_idx], dtype=torch.long).to(device)  # shape: (1, len(start_string))

    # Initialize the LSTM's hidden state
    state = model.init_hidden(input_idx.size(0), device)

    text_generated = []
    tqdm._instances.clear()

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(generation_length)):
            # Feed the current input (and hidden state) through the model
            predictions, state = model(input_idx, state, return_state=True)

            # Only the prediction for the most recently seen character matters
            # for picking the next one -> shape: (1, vocab_size)
            predictions = predictions[:, -1, :]

            # Sample the next character's index from the predicted probability
            # distribution (sampling, rather than always taking the top choice,
            # keeps the model from getting stuck in repetitive loops)
            input_idx = torch.multinomial(torch.softmax(predictions, dim=-1), num_samples=1)  # shape: (1, 1)

            # Convert the predicted index back to a character and store it
            text_generated.append(idx2char[input_idx.item()])

    return (start_string + ''.join(text_generated))

# Generate ~1000 characters of new ABC notation. Songs in this dataset start
# with an "X:" tune-number field, so "X" is a natural seed character.
generated_text = generate_text(model, start_string="X", generation_length=1000)
print(generated_text)

In [ ]:
# Extract individual songs from the generated block of text
generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
    # Synthesize the waveform from a generated song
    waveform = mdl.lab1.play_song(song)

    # If it's valid ABC notation (correct syntax), play it and save it
    if waveform:
        print("Generated song", i)
        ipythondisplay.display(waveform)

        numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
        wav_file_path = f"output_{i}.wav"
        write(wav_file_path, 88200, numeric_data)

        # Save the audio file to Comet -- find it under Audio / Assets & Artifacts
        experiment.log_asset(wav_file_path)

In [ ]:
# When you're done, close out the Comet experiment
experiment.end()